# 👁️ Smart Eye — Train model nhận diện có thêm cầu thang, cột điện, ổ gà...

Chạy được trên **Google Colab** hoặc **Kaggle Notebooks** (cả hai miễn phí, có GPU, chạy Linux nên export TFLite
được — **không cần WSL/Mac**). Colab hết giờ GPU → chuyển sang Kaggle (30 giờ GPU/tuần), notebook tự nhận ra môi trường.

**Quy trình:** tải 1 phần COCO (giữ người, xe...) → thêm dataset Roboflow / Mendeley / Kaggle + ảnh nhóm tự gán nhãn → gộp theo
`training/classes.yaml` → train YOLO → kiểm tra → xuất `smart_eye.tflite` cho app.

**Trước khi chạy:**
- *Colab:* *Runtime → Change runtime type → **T4 GPU***. Kết quả lưu trên Google Drive (`MyDrive/smart_eye`).
- *Kaggle:* thanh bên phải → *Session options* → **Accelerator: GPU T4** (hoặc P100), **Internet: On** (cần xác minh số
  điện thoại tài khoản Kaggle). Thêm dataset ổ gà: *Add Input* → tìm `andrewmvd/pothole-detection` → *Add*.
  Kết quả nằm ở `/kaggle/working/smart_eye` (tab *Output*); bấm *Save Version → Save & Run All* để chạy nền, tắt tab vẫn chạy.

Toàn bộ mất khoảng 1,5–3 giờ (phần lớn là train).

## 1. Chuẩn bị

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!pip -q install ultralytics fiftyone roboflow kaggle ai-edge-litert pyyaml

In [ ]:
REPO_BRANCH = 'feature/perf-and-classes'  # nhánh chứa thư mục training/ (đổi thành 'main' sau khi merge)
!rm -rf smart-eye && git clone -q -b $REPO_BRANCH https://github.com/langvietthanh/smart-eye.git
%cd smart-eye

import os
ON_KAGGLE = os.path.exists('/kaggle/working')
if ON_KAGGLE:
    WORK, DS = '/kaggle/working/smart_eye', '/kaggle/working/smart_eye_ds'
else:
    from google.colab import drive
    drive.mount('/content/drive')
    WORK, DS = '/content/drive/MyDrive/smart_eye', '/content/smart_eye_ds'
!mkdir -p $WORK/datasets $WORK/own_data $WORK/raw_photos $WORK/export

def secret(name):
    """Đọc secret: Colab (🔑 Secrets) hoặc Kaggle (Add-ons → Secrets); không có → None"""
    try:
        if ON_KAGGLE:
            from kaggle_secrets import UserSecretsClient
            return UserSecretsClient().get_secret(name)
        from google.colab import userdata
        return userdata.get(name)
    except Exception:
        return None
print('Môi trường:', 'Kaggle' if ON_KAGGLE else 'Colab', '· lưu tại', WORK)

## 2. Dữ liệu COCO (giữ các lớp cũ để model không "quên")
Chỉ tải ảnh có người, xe, ghế, biển báo... (~15 800 ảnh, ~3 GB). Bản v1 dùng 6000 ảnh → người / xe nhận kém hơn model gốc,
nên v2 tăng lên 15 000. Trên Colab chỉ tải 1 lần (lưu Drive); trên Kaggle tải lại mỗi lần chạy (~15–25 phút).

In [ ]:
import os
if not os.path.exists(f'{WORK}/datasets/coco_subset/dataset.yaml'):
    !python training/scripts/fetch_coco_subset.py --out $WORK/datasets/coco_subset --train 15000 --val 800
else:
    print('Đã có COCO subset trên Drive — bỏ qua')

## 3. Dataset có sẵn trên Roboflow Universe
1. Vào https://universe.roboflow.com, tìm: `pothole`, `stairs`, `utility pole`, `railing`, `manhole`, `bollard`,
   `traffic cone`, `curb`, `vietnam traffic sign`, `street vendor`...
2. Chọn dataset **giấy phép cho phép dùng** (VD CC BY 4.0), ảnh giống cảnh đường phố thật, nhiều ảnh.
3. Bấm **Download → YOLOv8 → show download code**, chép 3 giá trị `workspace`, `project`, `version` vào danh sách dưới.
4. API key: biểu tượng 🔑 (Secrets) bên trái Colab → thêm `ROBOFLOW_API_KEY`.

Tên lớp của dataset không cần khớp — `build_dataset.py` tự đổi qua `aliases` trong `classes.yaml`
(lớp lạ sẽ bị bỏ và được liệt kê trong báo cáo để bổ sung alias nếu cần).

In [ ]:
ROBOFLOW = [
    # ('workspace', 'project', version),
]

if ROBOFLOW:
    from roboflow import Roboflow
    rf = Roboflow(api_key=secret('ROBOFLOW_API_KEY'))
    for ws, proj, ver in ROBOFLOW:
        dest = f'{WORK}/datasets/rf_{proj}_v{ver}'
        if os.path.exists(dest):
            print('Đã có', dest); continue
        rf.workspace(ws).project(proj).version(ver).download('yolov8', location=dest)

## 3b. Dataset trên Mendeley Data
Điền `(id, phiên bản, tên)` lấy từ link `data.mendeley.com/datasets/<id>/<phiên bản>`.

- **BPID** — 161 ảnh ổ gà ở Bandung (Indonesia), 263 ổ gà, nắng / râm / ướt / ban đêm, giấy phép **CC BY 4.0**
  (phải ghi nguồn). Tác giả thiết kế làm **tập kiểm tra độc lập** → mặc định vào tập **val** (ảnh nằm trong thư mục `test`).
  Muốn dùng để **train**: đặt tên bắt đầu bằng `train_` (VD `'train_bpid'`).

In [ ]:
MENDELEY = [
    ('rgymy6dwdd', 1, 'bpid'),  # Bandung Pothole Image Dataset — DOI 10.17632/rgymy6dwdd.1
]
for ds_id, ver, name in MENDELEY:
    dest = f'{WORK}/datasets/{name}'
    if not os.path.exists(dest):
        !python training/scripts/fetch_mendeley.py $ds_id --version $ver --out "$dest"

## 3c. Dataset trên Kaggle
**Đang chạy trên Kaggle:** chỉ cần đã *Add Input* dataset ổ gà (xem đầu notebook) — không cần API key.

**Đang chạy trên Colab:**
1. Đăng nhập kaggle.com → ảnh đại diện → *Settings* → mục **API**.
2. Colab → biểu tượng 🔑 (Secrets) bên trái → thêm secret, bật *Notebook access*:
   - Nếu Kaggle cho tải file **`kaggle.json`** (*Create Legacy API Key*): mở file, thêm `KAGGLE_USERNAME` = `username`,
     `KAGGLE_KEY` = `key`.
   - Nếu Kaggle cấp **API token** dạng mới (*Generate New Token*): thêm `KAGGLE_API_TOKEN` = token đó.

- **Pothole Detection** (`andrewmvd/pothole-detection`) — 665 ảnh ổ gà, nhãn PASCAL VOC (XML, `build_dataset.py` tự đổi
  sang YOLO), giấy phép DbCL v1.0, ghi nguồn MakeML. Đặt tên `train_...` → dùng để **train** (BPID ở bước 3b để kiểm tra).

In [ ]:
KAGGLE = [
    ('andrewmvd/pothole-detection', 'train_kaggle_potholes'),
]
import shutil
for key in ('KAGGLE_API_TOKEN', 'KAGGLE_USERNAME', 'KAGGLE_KEY'):
    if secret(key):
        os.environ[key] = secret(key)  # Chỉ cần 1 kiểu: KAGGLE_API_TOKEN, hoặc KAGGLE_USERNAME + KAGGLE_KEY
import glob
for slug, name in KAGGLE:
    dest = f'{WORK}/datasets/{name}'
    if os.path.exists(dest):
        continue
    if ON_KAGGLE:
        # Kaggle có thể gắn ở /kaggle/input/<tên> hoặc sâu hơn (VD /kaggle/input/datasets/<chủ>/<tên>)
        found = [d for d in glob.glob('/kaggle/input/**/' + slug.split('/')[1], recursive=True) if os.path.isdir(d)]
        if not found:
            raise RuntimeError(f'Chưa Add Input dataset {slug}: thanh bên phải → Add Input → tìm "{slug}" → Add, rồi chạy lại.')
        shutil.copytree(found[0], dest)
        print('Đã lấy', slug, 'từ', found[0])
    else:
        !kaggle datasets download $slug -p "$dest" --unzip

## 4. Ảnh nhóm tự chụp (quan trọng nhất cho vỉa hè Việt Nam)
**Cách chụp:** điện thoại **dọc, đeo/cầm trước ngực** như khi dùng app, đi bộ trên vỉa hè thật; nhiều giờ trong ngày,
trời nắng/râm/mưa, cả ngày lẫn tối. Quay video rồi trích ảnh 1–2 ảnh/giây cũng được.

**4a. (Tuỳ chọn) Gán nhãn tự động:** chép ảnh vào `MyDrive/smart_eye/raw_photos/<tên đợt>/` rồi chạy ô dưới.
YOLO-World khoanh sẵn box → tải thư mục kết quả về, **sửa lại bằng CVAT / Label Studio / Roboflow**.

**4b. Ảnh đã gán nhãn xong:** export định dạng **YOLO** thành file `.zip`, chép vào `MyDrive/smart_eye/own_data/`.

In [ ]:
# 4a — gán nhãn tự động (bỏ qua nếu không có ảnh mới)
import glob
for batch in sorted(glob.glob(f'{WORK}/raw_photos/*/')):
    name = os.path.basename(batch.rstrip('/'))
    out = f'{WORK}/auto_labeled/{name}'
    if not os.path.exists(out):
        !python training/scripts/auto_label.py --images "$batch" --out "$out"
print('Kết quả gán nhãn tự động ở MyDrive/smart_eye/auto_labeled/ — nhớ kiểm tra & sửa trước khi đưa vào 4b')

In [ ]:
# 4b — giải nén dữ liệu đã gán nhãn xong
import zipfile
for z in sorted(glob.glob(f'{WORK}/own_data/*.zip')):
    dest = f'{WORK}/datasets/own_' + os.path.splitext(os.path.basename(z))[0]
    if not os.path.exists(dest):
        zipfile.ZipFile(z).extractall(dest)
        print('Giải nén', z, '→', dest)

## 5. Gộp dữ liệu theo `classes.yaml`
Đọc kỹ bảng thống kê: lớp có **⚠ thiếu dữ liệu** (< 300 vật) sẽ học kém → bổ sung ảnh cho lớp đó trước khi train lâu.

In [ ]:
def spec(d):
    base = os.path.basename(d.rstrip('/'))
    # Thư mục tên train_* / val_* → ép cả nguồn vào tập đó
    return d + ('#train' if base.startswith('train_') else '#val' if base.startswith('val_') else '')
sources = ' '.join(f'--source "{spec(d)}"' for d in sorted(glob.glob(f'{WORK}/datasets/*/')))
!python training/scripts/build_dataset.py $sources --out $DS

# Lớp bắt buộc phải có dữ liệu train — thiếu thì dừng luôn, không train "chay" 2 giờ
import json
REQUIRED = ['pothole']
stats = json.load(open(f'{DS}/stats.json', encoding='utf-8'))['instances']
missing = [c for c in REQUIRED if stats[c]['train'] == 0]
if missing:
    raise RuntimeError(f'Không có ảnh train cho: {missing} — kiểm tra bước 3b/3c (dataset đã tải / Add Input chưa?)')
print('Ảnh train lớp bắt buộc:', {c: stats[c]['train'] for c in REQUIRED})

## 6. Train
- `yolo11n.pt`: nhỏ, nhanh, hợp điện thoại (có thể đổi `yolov8n.pt`).
- `IMGSZ = 320` khớp app hiện tại; `256` nhanh hơn ~1,5 lần nhưng kém với vật nhỏ ở xa.
- ~16 000 ảnh × 80 epoch ≈ 3–4 giờ trên T4 (Kaggle cho 12 giờ / phiên). **Bị ngắt giữa chừng:** chạy lại bước 1, bước 5 (dataset gộp bị mất khi
  ngắt), rồi chạy ô dưới với `RESUME = True` — train tiếp từ epoch đang dở.
- **Đã train dở trên Colab, chuyển sang Kaggle:** tải `MyDrive/smart_eye/runs/smart_eye_v1/weights/last.pt` về, trên Kaggle
  *Add Input → Upload* file đó, rồi đặt `BASE = '/kaggle/input/<tên bạn đặt>/last.pt'` (để `RESUME = False`) —
  model học tiếp từ trọng số đã có thay vì từ đầu.

In [ ]:
BASE, IMGSZ, EPOCHS, BATCH = 'yolo11n.pt', 320, 80, 64
RUN_NAME, RESUME = 'smart_eye_v2', False
RUN_DIR = f'{WORK}/runs/{RUN_NAME}'

# Chạy bằng lệnh `yolo` (tiến trình riêng): pip install ở bước 1 có thể cài lại Pillow giữa phiên,
# import ultralytics trong kernel sẽ trộn 2 phiên bản PIL → ImportError '_Ink'
if RESUME:
    !yolo detect train resume model="{RUN_DIR}/weights/last.pt"
else:
    !yolo detect train model={BASE} data="{DS}/data.yaml" imgsz={IMGSZ} epochs={EPOCHS} batch={BATCH} patience=20 cos_lr=True close_mosaic=10 project="{WORK}/runs" name={RUN_NAME} exist_ok=True

## 7. Kiểm tra theo từng lớp

In [ ]:
# Bảng in ra có mAP50 từng lớp (dòng `pothole` = điểm ổ gà trên BPID)
!yolo detect val model="{RUN_DIR}/weights/best.pt" data="{DS}/data.yaml" imgsz={IMGSZ} plots=False
import json
stats = json.load(open(f'{DS}/stats.json', encoding='utf-8'))['instances']
print('Chưa có trong tập val (chưa đánh giá được):', [c for c, v in stats.items() if v['val'] == 0])

## 8. Xuất TFLite cho app
Xuất bản **FP32** (bản INT8 hiệu chỉnh ít ảnh bị "chặn trần" điểm tin cậy), rồi chuyển trọng số về định dạng
runtime Android/iOS đọc được, rồi chấm lại đúng như app chạy.

In [ ]:
import shutil
!yolo export model="{RUN_DIR}/weights/best.pt" format=tflite imgsz={IMGSZ}
tflite = max(glob.glob(f'{RUN_DIR}/weights/**/*.tflite', recursive=True), key=os.path.getmtime)
print('Export:', tflite)
!python tool/inline_tflite_buffers.py "$tflite"
!python tool/eval_model.py --model "$tflite" --data $DS/data.yaml --conf 0.25,0.35

out = f'{WORK}/export/smart_eye_{RUN_NAME}_{IMGSZ}.tflite'
shutil.copy(tflite, out)
print('Đã lưu:', out)
if ON_KAGGLE:
    print('Tải file ở tab Output bên phải: smart_eye/export/')
else:
    from google.colab import files
    files.download(out)

## 9. Đưa vào app
1. Đổi tên file vừa tải thành **`smart_eye.tflite`**, chép vào `assets/models/` của repo.
2. `flutter run` — app **tự dùng model mới** (tên lớp đọc từ metadata trong model, không cần sửa code).
   Log khởi động phải có: `Model: assets/models/smart_eye.tflite ... nhãn từ metadata`.
3. Thêm / đổi lớp: sửa `training/classes.yaml` **và** `lib/utils/label_catalog.dart` (tên tiếng Việt + nhóm nguy hiểm)
   — `flutter test` sẽ báo nếu 2 file lệch nhau.